In [ ]:
!pip install -q kaggle

ERROR: Operation cancelled by user


In [ ]:
!kaggle datasets files hannanh1610/rwf-2000

Next Page Token = CfDJ8LXGgraiXMNClPzcySXY5U7s8-dcDG115g-4oKb8yay8hbajHAIM1lyx6M4_DXBSFol96MGWVdTuQ717pHcFtkpgD0cG_Nf8-a9t0wxhU25PpTysduzUEOr9pqoYpWKf-yqoDWIy0WJs5i0YryQx1Wv5ykbSMVW1N44LoTL6tS_HyQ
name                                         size  creationDate                
----------------------------------------  -------  --------------------------  
raw/raw/train/Fight/train_Fight_0001.avi  2465594  2026-09-30 01:10:49.497000  
raw/raw/train/Fight/train_Fight_0002.avi  2538716  2026-09-30 01:10:49.590000  
raw/raw/train/Fight/train_Fight_0003.avi  2672260  2026-09-30 01:10:52.284000  
raw/raw/train/Fight/train_Fight_0004.avi  9476230  2026-09-30 01:10:49.850000  
raw/raw/train/Fight/train_Fight_0005.avi  9299134  2026-09-30 01:10:50.007000  
raw/raw/train/Fight/train_Fight_0006.avi  9486384  2026-09-30 01:10:50.474000  
raw/raw/train/Fight/train_Fight_0007.avi  8762910  2026-09-30 01:10:50.255000  
raw/raw/train/Fight/train_Fight_0008.avi  7276692  2026-09-30 01:10:52.880000  
raw

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if len(files) > 0:
        print(files[:3])

/kaggle/input


In [ ]:
!mkdir -p /content/RWF-2000

!kaggle datasets download \
    -d hannanh1610/rwf-2000 \
    -p /content/RWF-2000 \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/hannanh1610/rwf-2000
License(s): unknown
100% 12.4G/12.4G [01:57<00:00, 113MB/s]



In [ ]:
!find /content/RWF-2000 -maxdepth 3 -type d

/content/RWF-2000
/content/RWF-2000/skeleton_noback
/content/RWF-2000/skeleton_noback/skeleton_noback
/content/RWF-2000/skeleton_noback/skeleton_noback/val
/content/RWF-2000/skeleton_noback/skeleton_noback/train
/content/RWF-2000/raw
/content/RWF-2000/raw/raw
/content/RWF-2000/raw/raw/val
/content/RWF-2000/raw/raw/train


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os

In [ ]:
# Use output of OpenPose with or without background
BACKGROUND = False
# Paths to videos for training
PATHS = [
    "/content/RWF-2000/raw/raw",
    "/content/RWF-2000/skeleton_noback/skeleton_noback"
]
FRAME_FUNC = 'frame_diff'
# To use frame diff to weight t (current) or t+1
WEIGHT_CURRENT = True

In [ ]:
print(tf.config.list_physical_devices('GPU'))
print(tf.config.list_logical_devices('GPU'))

[]
[]


In [ ]:
SEED = 0
import random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Load data

In [ ]:
ORIGINAL_FRAMES_PER_VIDEO = 150
FRAMES_PER_VIDEO = 50 + 1
VIDEO_WIDTH, VIDEO_HEIGHT = 100, 100
N_CHANNELS = 3

def load_videos(video_IDs: list, video_frames: int = FRAMES_PER_VIDEO, video_width: int = VIDEO_WIDTH, video_height: int = VIDEO_HEIGHT,
                video_channels: int = N_CHANNELS, dtype = np.float32, normalize: bool = False) -> tuple:
    videos = np.empty((len(video_IDs), video_frames, video_height, video_width, video_channels), dtype=dtype)

    # Indexes of frames to be kept to comply with video_frames
    frames_idx = set(np.round(np.linspace(0, ORIGINAL_FRAMES_PER_VIDEO - 1, video_frames)).astype(int))

    for i, video_ID in enumerate(video_IDs):
        cap = cv2.VideoCapture(video_ID)
        frames = []
        index = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if index in frames_idx:
                frame = cv2.resize(frame, (video_width, video_height)).astype(dtype)
                if normalize:
                    frame /= 255.0
                frames.append(frame)
            index += 1
        cap.release()

        videos[i,] = np.array(frames)

    return videos

In [ ]:
import gc
from tensorflow.keras import backend as K

class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, video_IDs: list, video_labels: dict, batch_size: int, paths: list = [''], video_width: int = VIDEO_WIDTH, video_height: int = VIDEO_HEIGHT,
                video_frames: int = FRAMES_PER_VIDEO, video_channels: int = N_CHANNELS, dtype = np.float32, normalize: bool = False, shuffle: bool = True):
        self.video_IDs = video_IDs
        self.video_labels = video_labels
        self.batch_size = batch_size
        self.paths = paths
        self.video_width = video_width
        self.video_height = video_height
        self.video_frames = video_frames
        self.video_channels = video_channels
        self.dtype = dtype
        self.normalize = normalize
        self.shuffle = shuffle

    def __len__(self):
        return len(self.video_IDs) // self.batch_size

    def __getitem__(self, idx):
        batch_IDs = self.video_IDs[idx*self.batch_size:(idx+1)*self.batch_size]
        input_videos = []

        for index, path in enumerate(self.paths):
            if index == 0:
                # Raw videos
                batch_IDs_full_path = [path + ID for ID in batch_IDs]
            else:
                # Skeleton videos have "output_" prefix
                batch_IDs_full_path = [path + os.path.dirname(ID) + "/output_" + os.path.basename(ID)
                                      for ID in batch_IDs]

            videos = load_videos(batch_IDs_full_path, self.video_frames, self.video_width,
                                         self.video_height, self.video_channels, self.dtype, self.normalize)

            input_videos.append(videos)

        labels = np.array([self.video_labels[ID] for ID in batch_IDs])

        return tuple(input_videos), labels


    def on_epoch_end(self):
        'Updates indexes after each epoch'
        if self.shuffle:
            np.random.shuffle(self.video_IDs)
        # Clear memory after epochs
        gc.collect()
        #K.clear_session()

## Video to load

In [ ]:
def get_slug_from_path(path):
    """
    Function to get slug from path
    slug must contain /train or /val because there are repeated names
    """
    # Try train index first
    index = path.rfind('/train/')
    if index == -1:
        index = path.rfind('/val/')
    return path[index:]

In [ ]:
import glob

train_video_IDs = [get_slug_from_path(path) for path in glob.glob(PATHS[0]+'/train/*/*')]
test_video_IDs = [get_slug_from_path(path) for path in glob.glob(PATHS[0]+'/val/*/*')]

train_video_labels = {video: 0 if 'NonFight' in video else 1 for video in train_video_IDs}
test_video_labels = {video: 0 if 'NonFight' in video else 1 for video in test_video_IDs}


train_generator = DataGenerator(train_video_IDs, train_video_labels, batch_size=10, paths=PATHS)
test_generator = DataGenerator(test_video_IDs, test_video_labels, batch_size=10, paths=PATHS)

## Frame functions

In [ ]:
import tensorflow.keras.backend as K

def tf_frame_diff(video):
    return video[1:] - video[:-1]

def tf_frame_dist(video):
    video_diff = tf_frame_diff(video)
    return K.sqrt(K.sum(K.square(video_diff), axis=-1, keepdims=True))

if WEIGHT_CURRENT:
    def tf_frame_diff_dist_combined(video):
        video_diff = tf_frame_diff(video)
        video_diff_current = tf.nn.relu(-video_diff)
        video_diff_next = tf.nn.relu(video_diff)
        video_diff_next_norm = K.sqrt(K.sum(K.square(video_diff_next), axis=-1, keepdims=True))
        return K.concatenate([video_diff_current, video_diff_next_norm])
else:
    def tf_frame_diff_dist_combined(video):
        video_diff = tf_frame_diff(video)
        video_diff_current = tf.nn.relu(video_diff)
        video_diff_prev = tf.nn.relu(-video_diff)
        video_diff_prev_norm = K.sqrt(K.sum(K.square(video_diff_prev), axis=-1, keepdims=True))
        return K.concatenate([video_diff_current, video_diff_prev_norm])

frame_func_dict = {'frame_diff': tf_frame_diff, 'frame_dist': tf_frame_dist, 'frame_diff_dist_combined': tf_frame_diff_dist_combined}
frame_func = frame_func_dict[FRAME_FUNC]

In [ ]:
@tf.keras.utils.register_keras_serializable(package="custom")
class FrameFunc(tf.keras.layers.Layer):
    def __init__(self, mode="frame_diff", weight_current=True, **kwargs):
        super().__init__(**kwargs)
        self.mode = mode
        self.weight_current = weight_current

    def call(self, x):                       # x: (B, T, H, W, C)
        diff = x[:, 1:] - x[:, :-1]          # (B, T-1, H, W, C)
        if self.mode == "frame_diff":
            return diff
        if self.mode == "frame_dist":
            return tf.sqrt(tf.reduce_sum(tf.square(diff), axis=-1, keepdims=True))
        # frame_diff_dist_combined
        if self.weight_current:
            main, other = tf.nn.relu(-diff), tf.nn.relu(diff)
        else:
            main, other = tf.nn.relu(diff), tf.nn.relu(-diff)
        other_norm = tf.sqrt(tf.reduce_sum(tf.square(other), axis=-1, keepdims=True))
        return tf.concat([main, other_norm], axis=-1)

    def compute_output_shape(self, input_shape):
        b, t, h, w, c = input_shape
        out_c = {"frame_diff": c, "frame_dist": 1}.get(self.mode, c + 1)
        return (b, t - 1, h, w, out_c)

    def get_config(self):
        return {**super().get_config(),
                "mode": self.mode, "weight_current": self.weight_current}

In [ ]:
@tf.keras.utils.register_keras_serializable(package="custom")
class SelectFrames(tf.keras.layers.Layer):
    def __init__(self, drop_last=True, **kwargs):
        super().__init__(**kwargs)
        self.drop_last = drop_last

    def call(self, x):
        return x[:, :-1] if self.drop_last else x[:, 1:]

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[1] - 1, *input_shape[2:])

    def get_config(self):
        return {**super().get_config(), "drop_last": self.drop_last}

## ConvLSTM

In [ ]:
inputs_raw = tf.keras.layers.Input(shape=(FRAMES_PER_VIDEO, VIDEO_HEIGHT, VIDEO_WIDTH, N_CHANNELS))
inputs_openpose = tf.keras.layers.Input(shape=(FRAMES_PER_VIDEO, VIDEO_HEIGHT, VIDEO_WIDTH, N_CHANNELS))

inputs_diff = FrameFunc(mode=FRAME_FUNC, weight_current=WEIGHT_CURRENT)(inputs_raw)
inputs_to_weight = SelectFrames(drop_last=WEIGHT_CURRENT, name="select_openpose_frames")(inputs_openpose)

inputs_diff_norm = tf.keras.layers.BatchNormalization()(inputs_diff)
inputs_diff_time_info_weight = tf.keras.layers.ConvLSTM2D(filters=9, kernel_size=(3, 3), return_sequences=True, data_format='channels_last', activation='tanh')(inputs_diff_norm)

convolutional_layer = tf.keras.layers.Conv2D(filters=9, kernel_size=(3,3), activation='relu')
inputs_openpose_soft = tf.keras.layers.TimeDistributed(convolutional_layer)(inputs_to_weight)

inputs_openpose_norm = tf.keras.layers.BatchNormalization(scale=False, center=False)(inputs_openpose_soft)

inputs_weighted = tf.keras.layers.Add()([inputs_openpose_norm, inputs_diff_time_info_weight])

x = tf.keras.layers.ConvLSTM2D(filters=32, kernel_size=(3, 3), return_sequences=False, data_format='channels_last', activation='tanh')(inputs_weighted)

x = tf.keras.layers.DepthwiseConv2D(kernel_size=(3, 3), depth_multiplier=2, activation='relu', data_format='channels_last')(x)

x = tf.keras.layers.GlobalAveragePooling2D(data_format='channels_last')(x)

x = tf.keras.layers.Dense(units=128, activation='relu')(x)
x = tf.keras.layers.Dense(units=16, activation='relu')(x)
outputs = tf.keras.layers.Dense(units=1, activation='sigmoid')(x)

model = tf.keras.Model([inputs_raw, inputs_openpose], outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 51, 100,   │          0 │ -                 │
│ (InputLayer)        │ 100, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer         │ (None, 51, 100,   │          0 │ -                 │
│ (InputLayer)        │ 100, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ select_openpose_fr… │ (None, 50, 100,   │          0 │ input_layer_1[0]… │
│ (SelectFrames)      │ 100, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frame_func          │ (None, 50, 100,   │          0 │ input_layer[0][0] │
│ (FrameFunc)         │ 100, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 50, 98,    │        252 │ select_openpose_… │
│ (TimeDistributed)   │ 98, 9)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 50, 100,   │         12 │ frame_func[0][0]  │
│ (BatchNormalizatio… │ 100, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 98,    │         18 │ time_distributed… │
│ (BatchNormalizatio… │ 98, 9)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d         │ (None, 50, 98,    │      3,924 │ batch_normalizat… │
│ (ConvLSTM2D)        │ 98, 9)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 98,    │          0 │ batch_normalizat… │
│                     │ 98, 9)            │            │ conv_lstm2d[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_1       │ (None, 96, 96,    │     47,360 │ add[0][0]         │
│ (ConvLSTM2D)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 94, 94,    │        640 │ conv_lstm2d_1[0]… │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ depthwise_conv2d… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      8,320 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │      2,064 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         17 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 62,607 (244.56 KB)

 Trainable params: 62,583 (244.46 KB)

 Non-trainable params: 24 (96.00 B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model.load_weights("/content/drive/MyDrive/last_model.keras")

In [ ]:
best_model_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/best_model.keras',
    monitor='val_accuracy',
    mode='max',
    save_best_only=True
)

# Save model after every epoch -> cuối cùng sẽ là last_model.keras
last_model_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/last_model.keras',
    save_best_only=False,
    save_freq="epoch",
)

model_checkpoint_callback = [
    best_model_callback,
    last_model_callback
]

In [ ]:
history = model.fit(train_generator, initial_epoch=49, epochs=50, validation_data=test_generator, verbose=1, callbacks=model_checkpoint_callback)

Epoch 50/50
160/160 ━━━━━━━━━━━━━━━━━━━━ 881s 6s/step - accuracy: 0.9131 - loss: 0.2295 - val_accuracy: 0.8625 - val_loss: 0.4664


## Model performance

In [ ]:
import tensorflow as tf

old_model = tf.keras.models.load_model(
    "/content/best_model_89_50.keras",
    compile=True,
    custom_objects={
        "FrameFunc": FrameFunc,
        "SelectFrames": SelectFrames
    }
)
loss, acc = old_model.evaluate(test_generator)
print(f"Loss: {loss:.4f} | Accuracy: {acc:.4f}")

40/40 ━━━━━━━━━━━━━━━━━━━━ 206s 5s/step - accuracy: 0.8950 - loss: 0.4244
Loss: 0.4244 | Accuracy: 0.8950


In [ ]:
from unrolled_convlstm import convert_model

new_model = convert_model(old_model)

max|diff| model cu vs moi = 4.47e-08


In [ ]:
!pip install tf2onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 53.4 MB/s eta 0:00:00


In [ ]:
import tf2onnx, onnx

print("Thứ tự input:", [i.name for i in new_model.inputs])

spec = (tf.TensorSpec((1, 51, 100, 100, 3), tf.float32, name="input_0"),
        tf.TensorSpec((1, 51, 100, 100, 3), tf.float32, name="input_1"))

@tf.function(input_signature=spec)
def fn(a, b):
    return new_model([a, b])

tf2onnx.convert.from_function(fn, input_signature=spec, opset=13,
                              output_path="/content/model_unrolled.onnx")

m = onnx.load("/content/model_unrolled.onnx")
print("Input ONNX:", [i.name for i in m.graph.input])
print("Op types:", sorted({n.op_type for n in m.graph.node}))

Thứ tự input: ['input_layer', 'input_layer_1']
Input ONNX: ['input_0', 'input_1']
Op types: ['Add', 'Concat', 'Conv', 'GlobalAveragePool', 'MatMul', 'Mul', 'Relu', 'Sigmoid', 'Slice', 'Split', 'Squeeze', 'Sub', 'Tanh', 'Transpose', 'Unsqueeze']


In [ ]:
import onnxruntime as ort
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

sess = ort.InferenceSession("model_unrolled.onnx", providers=["CPUExecutionProvider"])
in_names = [i.name for i in sess.get_inputs()]

y_true, y_prob = [], []
for i in range(len(test_generator)):
    X_batch, y_batch = test_generator[i]
    feed = {n: x.astype(np.float32) for n, x in zip(in_names, X_batch)}
    p = sess.run(None, feed)[0]
    y_prob.append(p)
    y_true.append(y_batch)

y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)

if y_prob.ndim == 1 or y_prob.shape[-1] == 1:
    y_pred = (y_prob.ravel() > 0.5).astype(int)
else:
    y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=["NonFight", "Fight"], digits=4))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    NonFight     0.9115    0.8750    0.8929       200
       Fight     0.8798    0.9150    0.8971       200

    accuracy                         0.8950       400
   macro avg     0.8956    0.8950    0.8950       400
weighted avg     0.8956    0.8950    0.8950       400

[[175  25]
 [ 17 183]]
